# Natural language versus Python prompts for ADE knowledge graph construction

This notebook trains two otherwise matched QLoRA adapters and evaluates them with strict complete-triple micro F1. Select a Kaggle GPU accelerator before running.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', '/kaggle/working/nl-vs-pl-kg-construction'))
if not PROJECT_DIR.exists():
    raise FileNotFoundError('Add this repository as a Kaggle input, clone it into /kaggle/working, or set PROJECT_DIR.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements-kaggle.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR)], check=True)

## Attach the ADE splits

Upload or attach a Kaggle dataset containing `train_triples.json`, `val_triples.json`, and `test_triples.json`. Set `ADE_INPUT` to that directory. The files must follow `data/README.md`.

In [ ]:
ADE_INPUT = Path(os.environ.get('ADE_INPUT', '/kaggle/input/ade-codekgc'))
ADE_WORK = PROJECT_DIR / 'data' / 'ade'
ADE_WORK.mkdir(parents=True, exist_ok=True)
for name in ('train_triples.json', 'val_triples.json', 'test_triples.json'):
    source = ADE_INPUT / name
    if not source.exists():
        raise FileNotFoundError(f'Missing {source}. Attach the ADE split dataset or set ADE_INPUT.')
    shutil.copy2(source, ADE_WORK / name)
print('ADE files ready:', sorted(p.name for p in ADE_WORK.glob('*.json')))

## Validate the pipeline without a model

This checks the JSON schema, both parsers, and metric serialization before GPU work begins.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=PROJECT_DIR, check=True)
subprocess.run(['nl-pl-kgc', 'smoke-test', '--dataset', str(PROJECT_DIR / 'tests' / 'fixtures.json'), '--output-dir', str(PROJECT_DIR / 'outputs' / 'smoke-test')], cwd=PROJECT_DIR, check=True)

## Train the matched conditions

The default is 200 update steps as in the paper. For a quick integration check, temporarily lower `train_steps` in the YAML to 5; do not report that run as a replication.

In [ ]:
CONFIG = PROJECT_DIR / 'configs' / 'ade_mistral7b.yaml'
subprocess.run(['nl-pl-kgc', 'train', '--config', str(CONFIG), '--format', 'both'], cwd=PROJECT_DIR, check=True)

## Evaluate with strict triple-level micro F1

In [ ]:
for condition in ('natural', 'code'):
    adapter = PROJECT_DIR / 'outputs' / 'ade-mistral7b' / condition / 'adapter'
    subprocess.run(['nl-pl-kgc', 'evaluate', '--config', str(CONFIG), '--format', condition, '--adapter', str(adapter)], cwd=PROJECT_DIR, check=True)

In [ ]:
import json, pandas as pd
rows = []
for condition in ('natural', 'code'):
    path = PROJECT_DIR / 'outputs' / 'ade-mistral7b' / condition / 'metrics.json'
    rows.append({'condition': condition, **json.loads(path.read_text())})
pd.DataFrame(rows).set_index('condition')